# Legacy-Parity Sweeps: Ketamine + Psilocybin (AdEx Whole-Brain)

This notebook reproduces the **original TVBSim-style sweeps** with the cleaner `tvbtoolkit` API:
- Ketamine depth states (control, sub-anaesthesia, anaesthesia)
- Receptor-driven psilocybin gradients
- Complexity metrics: **LZc** and **PCI (Casali-like)**

All outputs are saved under a clear directory tree.

In [1]:
from pathlib import Path
import numpy as np
import scipy.io
import matplotlib.pyplot as plt

from tvbtoolkit import (
    WholeBrainConfig, OutputConfig, ConditionSpec,
    run_condition_batch,
    ketamine_depth_conditions, psilocybin_receptor_conditions,
    stimulation_schedule, build_stimulation_override,
    plot_example_timeseries, plot_metric_summary, detect_system_specs, recommend_parallel_workers, set_publication_style
)


PROJECT_ROOT = Path.cwd().resolve()
if (PROJECT_ROOT / 'src').exists():
    pass
elif (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'data'
CONNECTIVITY_68 = PROJECT_ROOT / 'data' / 'connectivity' / 'connectivity_68.zip'
if not CONNECTIVITY_68.exists():
    raise FileNotFoundError(f'Connectivity file not found: {CONNECTIVITY_68}')

## System + Parallelism Info

This notebook auto-detects hardware specs and recommends a safe CPU worker count.

Use `FAST_MODE=True` for rapid iteration, then set `FAST_MODE=False` for final parity figures.

Note: TVB/Brian2 execution here is CPU-based; integrated GPU is not used by default.

Batch runs default to the **temporal average monitor** for speed and reduced output size.

In [ ]:
specs = detect_system_specs()
print(specs)

FAST_MODE = True  # Set False for final parity outputs

FAST_CFG = {
    'run_sim': 4000.0,
    'cut_transient': 1500.0,
    't_analysis': 250.0,
    'lzc_n_seeds': 3,
    'pci_n_seeds': 3,
    'n_stims': 3,
    'save_timeseries': False,
    'n_jobs_cap': 5,
    'monitor_mode': 'temporal_average',
    'temporal_average_period_ms': 1.0,
}

FINAL_CFG = {
    'run_sim': 5000.0,
    'cut_transient': 2000.0,
    't_analysis': 300.0,
    'lzc_n_seeds': 6,
    'pci_n_seeds': 6,
    'n_stims': 5,
    'save_timeseries': True,
    'n_jobs_cap': None,
    'monitor_mode': 'temporal_average',
    'temporal_average_period_ms': 1.0,
}

RUN_CFG = FAST_CFG if FAST_MODE else FINAL_CFG

N_JOBS = recommend_parallel_workers(task='whole_brain_tvb')
if RUN_CFG['n_jobs_cap'] is not None:
    N_JOBS = min(N_JOBS, RUN_CFG['n_jobs_cap'])

print('Mode:', 'FAST' if FAST_MODE else 'FINAL')
print('Run config:', RUN_CFG)
print('Workers used:', N_JOBS)

In [ ]:
# Output roots
out_root = OutputConfig(Path('./outputs/legacy_parity_sweeps'))
ket_dir = OutputConfig(out_root.root / 'ketamine')
psi_dir = OutputConfig(out_root.root / 'psilocybin')
out_root

## 1) Base strict-parity AdEx whole-brain config

In [ ]:
# Legacy-inspired defaults
run_sim = RUN_CFG['run_sim']
cut_transient = RUN_CFG['cut_transient']
t_analysis = RUN_CFG['t_analysis']

base_cfg = WholeBrainConfig(
    model_family='adex_zerlaut',
    zerlaut_matteo=False,
    zerlaut_gk_gna=False,
    zerlaut_order=2,  # second-order: matches TVBSim default
    stochastic_integrator=True,
    monitor_mode=RUN_CFG['monitor_mode'],
    temporal_average_period_ms=RUN_CFG['temporal_average_period_ms'],
    simulation_length_ms=run_sim,
    dt_ms=0.1,
    conduction_speed=4.0,
    coupling_strength=0.3,
    connectivity_zip=str(CONNECTIVITY_68),
    parameter_overrides={
        'parameter_model': {
            'T': 40.0,
            'P_e': [-0.0498, 0.00506, -0.025, 0.0014, -0.00041, 0.0105, -0.036, 0.0074, 0.0012, -0.0407],
            'P_i': [-0.0514, 0.004, -0.0083, 0.0002, -0.0005, 0.0014, -0.0146, 0.0045, 0.0028, -0.0153],
            'initial_condition': {
                'E': [0.004, 0.004], 'I': [0.010, 0.010],
                'C_ee': [0.0, 0.0], 'C_ei': [0.0, 0.0], 'C_ii': [0.0, 0.0],
                'W_e': [50.0, 50.0], 'W_i': [0.0, 0.0],
                'noise': [0.0, 0.0],
            },
        },
        'parameter_simulation': {'save_time': 1000.0},
    },
)

lzc_seeds = list(range(RUN_CFG['lzc_n_seeds']))
pci_seeds = list(range(RUN_CFG['pci_n_seeds']))
stimtimes = stimulation_schedule(cut_transient=cut_transient, run_sim=run_sim, t_analysis=t_analysis, n_stims=RUN_CFG['n_stims'])
base_cfg

## 2) Ketamine depth sweep (LZc)

In [ ]:
ket_conditions = ketamine_depth_conditions()
ket_lzc = run_condition_batch(
    base_cfg=base_cfg,
    conditions=ket_conditions,
    seeds=lzc_seeds,
    output=ket_dir,
    post_stim_window=300,
    save_timeseries=RUN_CFG['save_timeseries'],
    n_jobs=N_JOBS,
    use_processes=True,
    show_progress=True,
    monitor_mode_default='temporal_average',
    temporal_average_period_ms=RUN_CFG['temporal_average_period_ms'],
)
plot_metric_summary(ket_lzc, save_path=ket_dir.figures_dir / 'ketamine_lzc_pci_casali_like.png')
plot_example_timeseries(ket_lzc, trim_first_ms=1000, save_path=ket_dir.figures_dir / 'ketamine_timeseries.png')
print('Saved ketamine figures to', ket_dir.figures_dir)

## Monitor Sampling Benchmark

Estimated per-run sample reduction using temporal-average monitoring.

In [ ]:
n_samples_raw = int(base_cfg.simulation_length_ms / base_cfg.dt_ms)
n_samples_tavg = int(base_cfg.simulation_length_ms / RUN_CFG['temporal_average_period_ms'])
reduction = n_samples_raw / max(n_samples_tavg, 1)
print('Raw samples per run:        ', n_samples_raw)
print('Temporal-avg samples per run:', n_samples_tavg)
print('Expected reduction factor:   ', f'{reduction:.1f}x')

## 3) Ketamine stimulated sweep (PCI (Casali-like) across stimulation times)

In [ ]:
ket_stim_conditions = []
for cond in ket_conditions:
    for i, stimtime in enumerate(stimtimes):
        stim_override = build_stimulation_override(stimval=0.3e-3, stimtime=float(stimtime), stimdur=50.0, stimregion=[8])
        merged = dict(cond.parameter_overrides)
        # shallow merge is enough here because we only append a new nested key
        merged.update(stim_override)
        ket_stim_conditions.append(
            ConditionSpec(
                name=f'{cond.name}_stim{i+1}',
                description=f'{cond.description}, stim={i+1}',
                parameter_overrides=merged,
            )
        )

ket_pci = run_condition_batch(
    base_cfg=base_cfg,
    conditions=ket_stim_conditions,
    seeds=pci_seeds,
    output=OutputConfig(ket_dir.root / 'stimulated'),
    post_stim_window=300,
    save_timeseries=RUN_CFG['save_timeseries'],
    n_jobs=N_JOBS,
    use_processes=True,
    show_progress=True,
    monitor_mode_default='temporal_average',
    temporal_average_period_ms=RUN_CFG['temporal_average_period_ms'],
)

# Aggregate by base state name for cleaner plotting
agg = {}
for base_name in ['control', 'ketamine_subanesthesia', 'ketamine_anesthesia']:
    keys = [k for k in ket_pci.keys() if k.startswith(base_name + '_')]
    agg[base_name] = {
        'lzc': np.concatenate([ket_pci[k]['lzc'] for k in keys]),
        'pci_casali_like': np.concatenate([ket_pci[k]['pci_casali_like'] for k in keys]),
        'time_ms_example': ket_pci[keys[0]]['time_ms_example'],
        'raw_example': ket_pci[keys[0]]['raw_example'],
    }

plot_metric_summary(agg, save_path=ket_dir.figures_dir / 'ketamine_stimulated_lzc_pci_casali_like.png')
print('Saved stimulated ketamine figure to', ket_dir.figures_dir)

## 4) Receptor-driven psilocybin gradients (LZc + PCI (Casali-like))

In [ ]:
# Local receptor dataset copied into the new repo
receptor_candidates = [
    DATA_DIR / 'receptors' / 'serotonin_receptors_PET_DesikanKilliany68.mat',
    PROJECT_ROOT / 'data' / 'receptors' / 'serotonin_receptors_PET_DesikanKilliany68.mat',
    Path.cwd() / 'data' / 'receptors' / 'serotonin_receptors_PET_DesikanKilliany68.mat',
    Path.cwd().parent / 'data' / 'receptors' / 'serotonin_receptors_PET_DesikanKilliany68.mat',
]
receptor_path = next((p for p in receptor_candidates if p.exists()), None)
if receptor_path is None:
    raise FileNotFoundError(f'Receptor file not found. Tried: {[str(p) for p in receptor_candidates]}')

receptor_mat = scipy.io.loadmat(str(receptor_path))
receptor_names = [n.item() for n in receptor_mat['receptor_names'].flatten()]
receptor_values = receptor_mat['receptors'].T

receptors = {name: vals for name, vals in zip(receptor_names, receptor_values)}
receptors['5HT2a_minus_5HT1a_aff_03'] = receptor_values[2] - 0.3 * receptor_values[0]
receptors['5HT2a_minus_5HT1a_aff_05'] = receptor_values[2] - 0.5 * receptor_values[0]
receptors['5HT2a_shuffled'] = np.random.default_rng(0).permutation(receptors['5HT2a'])
receptors['5HT2a_uniform'] = np.random.default_rng(1).uniform(np.min(receptors['5HT2a']), np.max(receptors['5HT2a']), size=receptors['5HT2a'].shape[0])

psi_conditions = psilocybin_receptor_conditions(receptors)
psi_lzc = run_condition_batch(
    base_cfg=WholeBrainConfig(
        **{**base_cfg.__dict__, 'zerlaut_gk_gna': True}
    ),
    conditions=psi_conditions,
    seeds=lzc_seeds,
    output=psi_dir,
    post_stim_window=300,
    save_timeseries=RUN_CFG['save_timeseries'],
    n_jobs=N_JOBS,
    use_processes=True,
    show_progress=True,
    monitor_mode_default='temporal_average',
    temporal_average_period_ms=RUN_CFG['temporal_average_period_ms'],
)

plot_metric_summary(psi_lzc, save_path=psi_dir.figures_dir / 'psilocybin_receptor_lzc_pci_casali_like.png')
plot_example_timeseries(psi_lzc, trim_first_ms=1000, save_path=psi_dir.figures_dir / 'psilocybin_receptor_timeseries.png')
print('Saved psilocybin figures to', psi_dir.figures_dir)

## 5) Optional focused PCI (Casali-like) comparison: Control vs 5HT2a profile

In [ ]:
focus_profiles = ['control', 'psilocybin_5HT2a']
focus = {k: psi_lzc[k] for k in focus_profiles if k in psi_lzc}
set_publication_style()
fig = plot_metric_summary(focus, save_path=psi_dir.figures_dir / 'psilocybin_focus_control_vs_5HT2a.png')
print('Saved focus figure to', psi_dir.figures_dir)

## 6) Quick table summary

In [ ]:
def summarize(metric_dict, label):
    print('\n===', label, '===')
    for cond, vals in metric_dict.items():
        print(f"{cond:35s}  LZc={np.mean(vals['lzc']):.4f}±{np.std(vals['lzc']):.4f}   PCI (Casali-like)={np.mean(vals['pci_casali_like']):.4f}±{np.std(vals['pci_casali_like']):.4f}")

summarize(ket_lzc, 'Ketamine (spontaneous)')
summarize(psi_lzc, 'Psilocybin receptor gradients')